In [12]:
import pandas as pd
import numpy as np

DATASETS = ["cars3d", "dsprites", 
            "mpi3d",
            "clevr", "iraven", "shapes3d"]
AVG_COL = "__avg__"
AVG_LABEL = "Avg."
# Diccionarios para despliegue (ajústalos a tu gusto)
MODEL_NAME_MAP = {
    "resnet18": "ResNet-18",
    "resnet18_mixer": "LATTICE(ResNet-18)",
    "resnet18_mixer_rp64_all_cases": "LATTICE(ResNet-18)-64",
    "resnet18_mixer_rp128_all_cases": "LATTICE(ResNet-18)-128",
    "resnet18_mixer_rp256_all_cases": "LATTICE(ResNet-18)-256",
    "resnet18_algebraic_non_iid": "LATTICE(ResNet-18)-ALG",
    "split": "AIN",
    "split_4x": "AIN (4x)",
    "split_resnet_mixer": "AIN + LICG ",
    "split_resnet_mixer_red_64": "AIN + LICG -64",
    "split_resnet_mixer_red_128": "AIN + LICG -128",
    "split_resnet_mixer_red_256": "AIN + LICG -256",
    "split_resnet_algebraic_non_iid": "AIN + LICG -ALG",
    "split_resnet_algebraic_iid": "AIN + LICG -ALG + IID",
    "split_resnet_algebraic_adv":"AIN + LICG - ALG + ADV(ALL)",
    "split_resnet_algebraic_non_iid_unpredictable_target_1":"AIN + LICG - ALG + ADV (1)",
    "split_resnet_algebraic_non_iid_unpredictable_target_2":"AIN + LICG - ALG + ADV (2)",
    "lattice_ain_alg_l_0.5": "LATTICE(AIN) - l:0.5",
    "lattice_ain_alg_l_0.75": "LATTICE(AIN) - l:0.75",
    "lattice_ain_alg_l_1.5": "LATTICE(AIN) - l:1.5",
    "lattice_ain_alg_l_2": "LATTICE(AIN) - l:2",
    "split_resnet_mixer_iid": "AIN + LICG +IID",
    "split_resnet_mixer_no_mixer": "AIN + LICG  - MIXER",
    "split_resnet_mixer_no_mixer_iid": "AIN + LICG  - MIXER + IID",
    "split_resnet_mixer_red_iid_64": "AIN + LICG -64 + IID",
    "split_resnet_mixer_red_iid_128": "AIN + LICG -128 + IID",
    "split_resnet_mixer_red_iid_256": "AIN + LICG -256 + IID" ,
    "split_resnet_mixer_all_cases": "AIN + LICG  [ALL]",
    "split_resnet_mixer_all_cases_iid": "AIN + LICG  [ALL] + IID",
    "crm_resnet18": "CRM(R18)",
    "crm_split_resnet": "CRM(AIN)",
    "ed": "ED",
    "ed_mixer_rp64": "LATTICE(ED)-64",
    "ed_mixer_rp128": "LATTICE(ED)-128",
    "ed_mixer_rp256": "LATTICE(ED)-256",
    "ed_algebraic_non_iid": "LATTICE(ED)-ALG",
}


# Modelos a ignorar
IGNORE_ARCHS = {
    "split_1",
    "split_2",
    "split_3",
    "split_4",
    *{
        f"split_resnet_mixer_s{s}_rp{rp}_all_cases"
        for s in [1, 2, 3, 4]
        for rp in [64, 128, 256]
    },
}

DATASET_NAME_MAP = {
    "cars3d": "C3D",
    "dsprites": "dSprites",
    "mpi3d": "MPI3D",
    "clevr": "CLEVR",
    "iraven": "I-RAVEN",
    "shapes3d": "Sh3D",
}

# Columnas de métricas (para excluirlas al definir "config")
METRIC_COLS = ["train_acc", "val_acc", "ood_val_0_acc", "test_acc",    "val_4cases_twonn_id",
    "val_4cases_topsim",
    "val_4cases_sv_auc",
    "val_4cases_pscore_mean",
    "val_4cases_hoyer_sparsity",
    "val_4cases_n_components_90pct",
    "val_4cases_embedding_dim",
    # metricas post-hoc de CRM. TIENEN que estar aqui: cualquier columna
    # que no este en METRIC_COLS pasa a formar parte de la clave de
    # configuracion, y como varian entre semillas cada run quedaria en su
    # propio grupo (n=1, std=NaN) sin ningun error visible.
    "final_test_crm_acc",
    "final_test_crm_naive_acc",
    "final_test_baseline_acc",
    "final_val_crm_acc",
    "final_val_crm_naive_acc",
    "final_val_baseline_acc",
]

DECIMALS = 2
BOLD_TOL = 1e-12  # tolerancia para empates/floating
def fmt(mean, std, decimals=DECIMALS):
    if pd.isna(mean):
        return ""
    if pd.isna(std):
        return f"{mean:.{decimals}f} (—)"
    return f"{mean:.{decimals}f} ({std:.{decimals}f})"


## Cálculo Tabla Resultados (ignorando ablation de MPI3D)

In [13]:
import pandas as pd
import numpy as np

# ============================================================
# CONFIG
# ============================================================
DECIMALS = 2
BOLD_TOL = 1e-12   # tolerancia para comparar empates
STAR_TOL = 1e-12   # tolerancia para el *

# Si quieres cambiar etiquetas visibles, hazlo acá
DISPLAY_NAME_MAP = MODEL_NAME_MAP.copy()

# Filas sintéticas agregadas por familia
AGG_FAMILIES = {
    "__lcig_resnet18__": {
        "label": MODEL_NAME_MAP.get("resnet18_mixer", "LATTICE(ResNet-18)"),
        "members": [
            #"resnet18_mixer",
            "resnet18_mixer_rp64_all_cases",
            "resnet18_mixer_rp128_all_cases",
            "resnet18_mixer_rp256_all_cases",
            "resnet18_algebraic_non_iid",
        ],
    },
    "__lcig_ain__": {
        "label": MODEL_NAME_MAP.get("split_resnet_mixer", "LATTICE(AIN)"),
        "members": [
            "split_resnet_mixer_red_64",
            "split_resnet_mixer_red_128",
            "split_resnet_mixer_red_256",
            "split_resnet_algebraic_non_iid",
        ],
    },
    "__lcig_ed__": {
        "label": "LCIG(ED)",
        "members": [
            "ed_mixer_rp64",
            "ed_mixer_rp128",
            "ed_mixer_rp256",
            "ed_algebraic_non_iid",
        ],
    },
}

DISPLAY_NAME_MAP.update({
    "__lcig_resnet18__": AGG_FAMILIES["__lcig_resnet18__"]["label"],
    "__lcig_ain__": AGG_FAMILIES["__lcig_ain__"]["label"],
    "__lcig_ed__": AGG_FAMILIES["__lcig_ed__"]["label"],
})

# Orden de filas visibles en la tabla final

LATTICE_AIN = [
    "lattice_ain_alg_l_0.5",
    "lattice_ain_alg_l_0.75",
    "lattice_ain_alg_l_1.5",
    "lattice_ain_alg_l_2",
]
ROW_GROUPS = [
    ["resnet18", "__lcig_resnet18__", "crm_resnet18"],
    ["split", "__lcig_ain__"] + LATTICE_AIN + ["crm_split_resnet"],
    ["ed", "__lcig_ed__"],
]

# ============================================================
# HELPERS
# ============================================================
# Metrica por arquitectura. Las filas de CRM se reportan con su propia regla de
# decision (argmax conjunto con el sesgo extrapolado B*), no con el argmax por
# atributo. Lo que no aparezca aqui usa el argumento `metric`.
METRIC_BY_ARCH = {
    "crm_resnet18": "final_test_crm_acc",
    "crm_split_resnet": "final_test_crm_acc",
    "crm_split_resnet_mixer": "final_test_crm_acc",
}

def metric_for(arch, default):
    return METRIC_BY_ARCH.get(arch, default)

def safe_score(mean, std):
    """Criterio de selección: mean - std; si std es NaN, penalización 0."""
    if pd.isna(mean):
        return np.nan
    return mean - (0.0 if pd.isna(std) else std)

def build_cell(mean, std, decimals=2):
    return fmt(mean, std, decimals=decimals)

def build_md_cell(text, bold=False, star=False):
    if text == "":
        return text
    if star:
        text = text + "*"
    if bold:
        text = f"**{text}**"
    return text

def build_tex_cell(text, bold=False, star=False):
    if text == "":
        return text
    if star:
        text = text + r"$^\ast$"
    if bold:
        text = r"\textbf{" + text + "}"
    return text

def build_avg_cell(mean, decimals=2):
    if pd.isna(mean):
        return ""
    return f"{mean:.{decimals}f}"
    
def plot_table_results(method, show_individual=False, metric="test_acc",
                       per_arch_metric=True, drop_collapsed=False,
                       collapse_metric="train_acc", collapse_threshold=5.0):
    """metric: columna a tabular.

    "test_acc" es el argmax por atributo (la regla del baseline). Para la
    regla propia de CRM usa "final_test_crm_acc"; "final_test_crm_naive_acc"
    es el control de la ablacion (con B_hat en vez de B*).

    per_arch_metric=True (por defecto) hace que cada arquitectura use su
    metrica nativa segun METRIC_BY_ARCH: los baselines el argmax por atributo
    y las filas de CRM su regla propia. Ponlo en False para la tabla de
    ablacion, donde TODAS las filas se tabulan con la misma columna y lo
    unico que cambia entre ellas es el entrenamiento.

    drop_collapsed descarta las semillas que fallaron al entrenar. El corte se
    hace sobre `collapse_metric` (train_acc por defecto), NO sobre la metrica
    reportada: filtrar por el resultado seria seleccionar sobre la variable
    dependiente e inflaria la media. Un run colapsado no ajusta ni el train
    (train_acc < 3 frente a >25 en los sanos), asi que el criterio separa sin
    ambiguedad. Va desactivado por defecto; si lo activas, reporta cuantas
    semillas cayeron por modelo -- se imprimen al final.
    """
    dropped = []
    # ============================================================
    # 1) Mejor config por arch y dataset usando score = mean - std
    # ============================================================
    records = []

    for dataset in DATASETS:
        df = pd.read_pickle(f"{dataset}_{method}.pkl").copy()

        if "arch" not in df.columns or "seed" not in df.columns:
            raise ValueError(f"{dataset}_{method}.pkl debe contener columnas: arch, seed")

        non_metric_cols = [c for c in df.columns if c not in METRIC_COLS]
        config_cols = [c for c in non_metric_cols if c != "seed"]

        # Cada arquitectura se tabula con SU metrica: los baselines con el
        # argmax por atributo (test_acc) y las filas de CRM con su regla propia
        # (final_test_crm_acc). Hay que agregar por separado porque son columnas
        # distintas; si se hiciera de una sola vez, la columna de CRM seria NaN
        # para los baselines y el idxmax de mas abajo reventaria con
        # "encountered all NA values in a group".
        for arch, sub in df.groupby("arch", dropna=False):
            col = metric_for(arch, metric) if per_arch_metric else metric
            if col not in sub.columns or sub[col].notna().sum() == 0:
                print(f"[aviso] {dataset}: '{arch}' no tiene datos en '{col}', se omite")
                continue

            if drop_collapsed and collapse_metric in sub.columns:
                bad = sub[collapse_metric] < collapse_threshold
                if bad.any():
                    dropped.append((dataset, arch, int(bad.sum()), int(len(sub))))
                    sub = sub[~bad]
                if sub.empty:
                    print(f"[aviso] {dataset}: '{arch}' sin semillas tras "
                          f"descartar colapsadas, se omite")
                    continue

            per_seed = (
                sub.groupby(config_cols + ["seed"], dropna=False)[col]
                .mean()
                .reset_index()
            )

            stats = (
                per_seed.groupby(config_cols, dropna=False)[col]
                .agg(mean="mean", std="std", n="count")
                .reset_index()
            )

            stats["score"] = stats.apply(lambda r: safe_score(r["mean"], r["std"]), axis=1)
            if stats["score"].notna().sum() == 0:
                print(f"[aviso] {dataset}: '{arch}' sin score valido, se omite")
                continue

            r = stats.loc[stats["score"].idxmax()]
            records.append({
                "arch": arch,
                "dataset": dataset,
                "mean": r["mean"],
                "std": r["std"],
                "score": r["score"],
                "metric": col,
            })

    if dropped:
        total = sum(d[2] for d in dropped)
        print(f"[colapsadas] descartadas {total} semillas "
              f"({collapse_metric} < {collapse_threshold}):")
        for ds_, arch_, n_bad, n_tot in dropped:
            print(f"    {ds_:<10} {arch_:<32} {n_bad}/{n_tot}")

    long_df = pd.DataFrame(records)

    # ============================================================
    # 2) Construir filas sintéticas LCIG(X)
    # ============================================================
    agg_records = []

    for agg_arch, spec in AGG_FAMILIES.items():
        fam_df = long_df[long_df["arch"].isin(spec["members"])].copy()
        if fam_df.empty:
            continue

        for dataset in DATASETS:
            sub = fam_df[fam_df["dataset"] == dataset].copy()
            if sub.empty:
                continue

            best_row = sub.loc[sub["score"].idxmax()]
            agg_records.append({
                "arch": agg_arch,
                "dataset": dataset,
                "mean": best_row["mean"],
                "std": best_row["std"],
                "score": best_row["score"],
                "source_arch": best_row["arch"],
            })

    agg_df = pd.DataFrame(agg_records)

    all_df = pd.concat([long_df, agg_df], ignore_index=True, sort=False)

    # ============================================================
    # 3) Elegir qué filas mostrar
    # ============================================================
    if show_individual:
        visible_row_groups = [
            [a for arch in group
               for a in ([arch] + AGG_FAMILIES[arch]["members"]
                         if arch in AGG_FAMILIES else [arch])]
            for group in ROW_GROUPS
        ]
    else:
        visible_row_groups = ROW_GROUPS

    visible_arches = [a for group in visible_row_groups for a in group]
    print(visible_arches)
    visible_df = all_df[all_df["arch"].isin(visible_arches)].copy()

    mean_df = visible_df.pivot(index="arch", columns="dataset", values="mean").reindex(
        index=visible_arches, columns=DATASETS
    )
    std_df = visible_df.pivot(index="arch", columns="dataset", values="std").reindex(
        index=visible_arches, columns=DATASETS
    )
    score_df = visible_df.pivot(index="arch", columns="dataset", values="score").reindex(
        index=visible_arches, columns=DATASETS
    )

    mean_df = visible_df.pivot(index="arch", columns="dataset", values="mean").reindex(
    index=visible_arches, columns=DATASETS
    )
    std_df = visible_df.pivot(index="arch", columns="dataset", values="std").reindex(
        index=visible_arches, columns=DATASETS
    )
    score_df = visible_df.pivot(index="arch", columns="dataset", values="score").reindex(
        index=visible_arches, columns=DATASETS
    )
    
    TABLE_COLS = DATASETS + [AVG_COL]
    
    mean_df[AVG_COL] = mean_df[DATASETS].mean(axis=1, skipna=True)
    std_df[AVG_COL] = np.nan
    
    # Para la columna Avg. rankeamos por accuracy promedio, no por mean - std
    score_df[AVG_COL] = mean_df[AVG_COL]
    # ============================================================
    # 4) Bold:
    #    mantener la lógica original SOLO para baseline vs LCIG agregado
    # ============================================================
    summary_arches = [a for group in ROW_GROUPS for a in group]
    summary_df = all_df[all_df["arch"].isin(summary_arches)]
    
    summary_score_df = summary_df.pivot(
        index="arch", columns="dataset", values="score"
    ).reindex(index=summary_arches, columns=DATASETS)
    
    summary_mean_df = summary_df.pivot(
        index="arch", columns="dataset", values="mean"
    ).reindex(index=summary_arches, columns=DATASETS)
    
    summary_score_df[AVG_COL] = summary_mean_df[DATASETS].mean(axis=1, skipna=True)
    
    best_summary_score = summary_score_df[TABLE_COLS].max(axis=0, skipna=True)
    
    is_bold = pd.DataFrame(False, index=visible_arches, columns=TABLE_COLS)
    for arch in summary_arches:
        if arch in is_bold.index:
            is_bold.loc[arch] = score_df.loc[arch].sub(best_summary_score).abs() <= BOLD_TOL

    # ============================================================
    # 5) Star:
    #    máximo absoluto entre TODOS los modelos disponibles
    # ============================================================
    all_score_df = all_df.pivot(index="arch", columns="dataset", values="score").reindex(columns=DATASETS)
    all_mean_df = all_df.pivot(index="arch", columns="dataset", values="mean").reindex(columns=DATASETS)
    
    all_score_df[AVG_COL] = all_mean_df[DATASETS].mean(axis=1, skipna=True)
    
    best_global_score = all_score_df[TABLE_COLS].max(axis=0, skipna=True)
    is_star = score_df[TABLE_COLS].sub(best_global_score, axis=1).abs() <= STAR_TOL

    # ============================================================
    # 6) Tabla base
    # ============================================================
    plain = pd.DataFrame(index=visible_arches, columns=TABLE_COLS, dtype=object)
    
    for ds in DATASETS:
        for arch in visible_arches:
            plain.loc[arch, ds] = build_cell(
                mean_df.loc[arch, ds],
                std_df.loc[arch, ds],
                decimals=DECIMALS,
            )
    
    for arch in visible_arches:
        plain.loc[arch, AVG_COL] = build_avg_cell(
            mean_df.loc[arch, AVG_COL],
            decimals=DECIMALS,
        )
    # ============================================================
    # 7) NOTEBOOK DISPLAY
    # ============================================================
    plain_nb = plain.copy()

    for ds in DATASETS:
        for arch in visible_arches:
            txt = plain_nb.loc[arch, ds]
            if txt != "" and bool(is_star.loc[arch, ds]):
                plain_nb.loc[arch, ds] = txt + "*"

    plain_nb = plain_nb.rename(index=DISPLAY_NAME_MAP, columns=DATASET_NAME_MAP)

    is_bold_nb = (
        is_bold.rename(index=DISPLAY_NAME_MAP, columns=DATASET_NAME_MAP)
        .reindex(index=plain_nb.index, columns=plain_nb.columns)
        .fillna(False)
    )

    display(
        plain_nb.style.apply(
            lambda col: [
                "font-weight: bold" if bool(is_bold_nb.loc[idx, col.name]) else ""
                for idx in col.index
            ],
            axis=0,
        )
    )

    # ============================================================
    # 8) MARKDOWN
    # ============================================================
    COLUMN_NAME_MAP = {
        **DATASET_NAME_MAP,
        AVG_COL: AVG_LABEL,
    }
    md = plain.copy()

    for ds in DATASETS:
        for arch in visible_arches:
            md.loc[arch, ds] = build_md_cell(
                md.loc[arch, ds],
                bold=bool(is_bold.loc[arch, ds]),
                star=bool(is_star.loc[arch, ds]),
            )

    md = md.rename(index=DISPLAY_NAME_MAP, columns=COLUMN_NAME_MAP)
    print(md.to_markdown())

    # ============================================================
    # 9) LATEX
    # ============================================================
    tex = plain.copy()

    for ds in DATASETS:
        for arch in visible_arches:
            tex.loc[arch, ds] = build_tex_cell(
                tex.loc[arch, ds],
                bold=bool(is_bold.loc[arch, ds]),
                star=bool(is_star.loc[arch, ds]),
            )

    tex = tex.rename(index=DISPLAY_NAME_MAP, columns=COLUMN_NAME_MAP)

    latex_cols = DATASETS + [AVG_COL]
    dataset_headers = [COLUMN_NAME_MAP.get(ds, ds) for ds in latex_cols]
    colspec = "l" + "c" * len(latex_cols)

    latex_lines = []
    latex_lines.append(r"\begin{table}[t]")
    latex_lines.append(r"\centering")
    latex_lines.append(r"\small")
    latex_lines.append(r"\setlength{\tabcolsep}{4pt}")
    latex_lines.append(r"\renewcommand{\arraystretch}{1.1}")
    latex_lines.append(r"\resizebox{\textwidth}{!}{%")
    latex_lines.append(rf"\begin{{tabular}}{{{colspec}}}")
    latex_lines.append(r"\toprule")
    latex_lines.append("Dataset & " + " & ".join(dataset_headers) + r" \\")
    latex_lines.append(r"\midrule")

    for g_idx, group in enumerate(visible_row_groups):
        visible_rows_in_group = [arch for arch in group if arch in plain.index]
        if not visible_rows_in_group:
            continue

        for arch in visible_rows_in_group:
            row_name = DISPLAY_NAME_MAP.get(arch, arch)
            cells = [tex.loc[row_name, COLUMN_NAME_MAP.get(ds, ds)] for ds in latex_cols]
            latex_lines.append(row_name + " & " + " & ".join(cells) + r" \\")

        if g_idx < len(visible_row_groups) - 1:
            latex_lines.append(r"\midrule")

    latex_lines.append(r"\bottomrule")
    latex_lines.append(r"\end{tabular}")
    latex_lines.append(r"}")
    latex_lines.append(r"\caption{TODO: caption}")
    latex_lines.append(r"\label{tab:oracle_results}")
    latex_lines.append(r"\end{table}")

    latex_small = "\n".join(latex_lines)
    print(latex_small)

In [14]:
plot_table_results("id",show_individual=True,drop_collapsed=True)

[colapsadas] descartadas 7 semillas (train_acc < 5.0):
    cars3d     crm_split_resnet                 2/5
    dsprites   crm_split_resnet                 1/5
    mpi3d      crm_split_resnet                 2/5
    iraven     crm_split_resnet                 1/5
    shapes3d   crm_split_resnet                 1/5
['resnet18', '__lcig_resnet18__', 'resnet18_mixer_rp64_all_cases', 'resnet18_mixer_rp128_all_cases', 'resnet18_mixer_rp256_all_cases', 'resnet18_algebraic_non_iid', 'crm_resnet18', 'split', '__lcig_ain__', 'split_resnet_mixer_red_64', 'split_resnet_mixer_red_128', 'split_resnet_mixer_red_256', 'split_resnet_algebraic_non_iid', 'lattice_ain_alg_l_0.5', 'lattice_ain_alg_l_0.75', 'lattice_ain_alg_l_1.5', 'lattice_ain_alg_l_2', 'crm_split_resnet', 'ed', '__lcig_ed__', 'ed_mixer_rp64', 'ed_mixer_rp128', 'ed_mixer_rp256', 'ed_algebraic_non_iid']


,C3D,dSprites,MPI3D,CLEVR,I-RAVEN,Sh3D,__avg__
ResNet-18,33.12 (0.64),21.30 (0.87),43.65 (1.38),20.40 (5.29),11.49 (3.65),84.78 (1.42),35.79
LATTICE(ResNet-18),52.50 (1.35),22.13 (1.72),45.14 (1.41),41.03 (2.40),25.19 (7.15),89.81 (2.58),45.97
LATTICE(ResNet-18)-64,38.33 (1.88),17.50 (1.76),44.38 (1.73),36.14 (4.49),12.63 (3.91),83.50 (2.66),38.74
LATTICE(ResNet-18)-128,40.66 (1.50),22.13 (1.72),44.97 (2.16),36.70 (5.92),15.07 (2.57),84.66 (2.78),40.70
LATTICE(ResNet-18)-256,41.39 (1.48),21.74 (3.36),45.14 (1.41),41.03 (2.40),17.30 (3.27),85.78 (1.83),42.06
LATTICE(ResNet-18)-ALG,52.50 (1.35),22.98 (2.60),45.42 (2.21),37.02 (6.56),25.19 (7.15),89.81 (2.58),45.49
CRM(R18),55.36 (1.05),6.70 (2.70),32.79 (10.89),62.41 (3.59),8.39 (3.34),21.96 (5.40),31.27
AIN,44.16 (1.54),60.49 (2.32),56.76 (1.77),55.49 (4.95),64.17 (4.49),85.19 (5.39),61.04
AIN + LICG,54.33 (2.73),67.36 (2.71),54.59 (1.12),62.97 (3.04),71.41 (7.91),92.24 (1.80),67.15
AIN + LICG -64,50.83 (0.88),65.69 (2.80),56.61 (5.00),55.06 (4.33),65.38 (2.42),92.24 (1.80),64.30


|                        | C3D               | dSprites          | MPI3D            | CLEVR            | I-RAVEN          | Sh3D             |   Avg. |
|:-----------------------|:------------------|:------------------|:-----------------|:-----------------|:-----------------|:-----------------|-------:|
| ResNet-18              | 33.12 (0.64)      | 21.30 (0.87)      | 43.65 (1.38)     | 20.40 (5.29)     | 11.49 (3.65)     | 84.78 (1.42)     |  35.79 |
| LATTICE(ResNet-18)     | 52.50 (1.35)      | 22.13 (1.72)      | 45.14 (1.41)     | 41.03 (2.40)     | 25.19 (7.15)     | 89.81 (2.58)     |  45.97 |
| LATTICE(ResNet-18)-64  | 38.33 (1.88)      | 17.50 (1.76)      | 44.38 (1.73)     | 36.14 (4.49)     | 12.63 (3.91)     | 83.50 (2.66)     |  38.74 |
| LATTICE(ResNet-18)-128 | 40.66 (1.50)      | 22.13 (1.72)      | 44.97 (2.16)     | 36.70 (5.92)     | 15.07 (2.57)     | 84.66 (2.78)     |  40.7  |
| LATTICE(ResNet-18)-256 | 41.39 (1.48)      | 21.74 (3.36)      | 45.14 (1.41)     | 41

In [4]:
dataset="cars3d"
method="id"
df = pd.read_pickle(f"{dataset}_{method}.pkl").copy()

In [5]:
df['arch'].unique()

<StringArray>
[                            'split_resnet_mixer_alg_s1',
                        'split_resnet_mixer_red_iid_128',
                                         'ed_mixer_rp64',
                            'split_resnet_algebraic_adv',
                        'split_resnet_algebraic_non_iid',
                                                 'split',
                                  'ed_algebraic_non_iid',
                             'split_resnet_mixer_alg_s2',
                                              'resnet18',
                                        'ed_mixer_rp128',
                                 'lattice_ain_alg_l_0.5',
                             'split_resnet_mixer_alg_s4',
                             'split_resnet_mixer_alg_s3',
                         'resnet18_mixer_rp64_all_cases',
 'split_resnet_algebraic_non_iid_unpredictable_target_1',
                                                    'ed',
                           'split_resnet_mixer_no_mixer',


In [26]:
df[df['arch'] == "lattice_ain_alg_l_0.5"]

,arch,iso,n_epoch,seed,combination,train_acc,val_acc,ood_val_0_acc,test_acc,val_4cases_twonn_id,val_4cases_n_components_90pct,val_4cases_topsim,val_4cases_pscore_mean,val_4cases_sv_auc,val_4cases_hoyer_sparsity,val_4cases_embedding_dim,c
40,lattice_ain_alg_l_0.5,NaN,NaN,1,NaN,98.761059,94.637360,100.0,50.645633,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
41,lattice_ain_alg_l_0.5,NaN,NaN,2,NaN,98.594203,95.184235,100.0,52.491385,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
42,lattice_ain_alg_l_0.5,NaN,NaN,3,NaN,98.904568,94.559235,100.0,52.149971,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1


In [7]:
df['arch'].unique()

<StringArray>
[                            'split_resnet_mixer_alg_s1',
                        'split_resnet_mixer_red_iid_128',
                                         'ed_mixer_rp64',
                            'split_resnet_algebraic_adv',
                        'split_resnet_algebraic_non_iid',
                                                 'split',
                                  'ed_algebraic_non_iid',
                             'split_resnet_mixer_alg_s2',
                                              'resnet18',
                                        'ed_mixer_rp128',
                                 'lattice_ain_alg_l_0.5',
                             'split_resnet_mixer_alg_s4',
                             'split_resnet_mixer_alg_s3',
                         'resnet18_mixer_rp64_all_cases',
 'split_resnet_algebraic_non_iid_unpredictable_target_1',
                                                    'ed',
                           'split_resnet_mixer_no_mixer',


In [28]:
plot_table_results("id", show_individual=False)

,C3D,dSprites,MPI3D,CLEVR,I-RAVEN,Sh3D
ResNet-18,33.12 (0.64),21.30 (0.87),43.65 (1.38),20.40 (5.29),11.49 (3.65),84.78 (1.42)
LICG(ResNet-18),52.50 (1.35),22.13 (1.72),45.14 (1.41),41.03 (2.40),25.19 (7.15),89.81 (2.58)
AIN,44.16 (1.54),60.49 (2.32),56.76 (1.77),55.49 (4.95),64.17 (4.49),85.19 (5.39)
LCIG(AIN),54.33 (2.73)*,67.36 (2.71)*,54.59 (1.12),62.97 (3.04)*,71.41 (7.91),92.24 (1.80)
ED,48.66 (0.91),62.80 (1.08),66.96 (5.63),56.07 (3.26),72.41 (6.39),95.62 (1.73)
LCIG(ED),49.34 (0.79),56.88 (1.80),64.59 (6.12),61.13 (2.64),72.50 (3.40)*,96.72 (2.41)*


|                 | C3D               | dSprites          | MPI3D            | CLEVR             | I-RAVEN           | Sh3D              |
|:----------------|:------------------|:------------------|:-----------------|:------------------|:------------------|:------------------|
| ResNet-18       | 33.12 (0.64)      | 21.30 (0.87)      | 43.65 (1.38)     | 20.40 (5.29)      | 11.49 (3.65)      | 84.78 (1.42)      |
| LICG(ResNet-18) | 52.50 (1.35)      | 22.13 (1.72)      | 45.14 (1.41)     | 41.03 (2.40)      | 25.19 (7.15)      | 89.81 (2.58)      |
| AIN             | 44.16 (1.54)      | 60.49 (2.32)      | 56.76 (1.77)     | 55.49 (4.95)      | 64.17 (4.49)      | 85.19 (5.39)      |
| LCIG(AIN)       | **54.33 (2.73)*** | **67.36 (2.71)*** | 54.59 (1.12)     | **62.97 (3.04)*** | 71.41 (7.91)      | 92.24 (1.80)      |
| ED              | 48.66 (0.91)      | 62.80 (1.08)      | **66.96 (5.63)** | 56.07 (3.26)      | 72.41 (6.39)      | 95.62 (1.73)      |
| LCIG(ED)        | 49.34 (

In [19]:
plot_table_results("id_hoyer", show_individual=False)

,C3D,dSprites,MPI3D,CLEVR,I-RAVEN,Sh3D
ResNet-18,33.12 (0.64),21.36 (0.95),43.65 (1.38),16.54 (3.69),13.72 (2.27),82.82 (1.83)
LICG(ResNet-18),41.38 (1.48),22.13 (1.72),45.14 (1.41),40.76 (2.09),17.50 (2.94),85.65 (1.79)
AIN,44.16 (1.54),57.47 (1.91),56.71 (1.76),46.64 (5.31),65.41 (3.10),83.75 (2.39)
LCIG(AIN),50.83 (0.88)*,65.69 (2.80)*,54.59 (1.12),57.63 (3.32),71.29 (4.61),92.08 (2.95)
ED,48.64 (0.93),62.71 (1.33),66.96 (5.63)*,58.16 (3.02),75.76 (8.08)*,94.74 (1.76)
LCIG(ED),49.34 (0.79),48.27 (3.32),59.92 (5.74),61.83 (2.43)*,75.11 (7.54),96.60 (2.19)*


|                 | C3D               | dSprites          | MPI3D             | CLEVR             | I-RAVEN           | Sh3D              |
|:----------------|:------------------|:------------------|:------------------|:------------------|:------------------|:------------------|
| ResNet-18       | 33.12 (0.64)      | 21.36 (0.95)      | 43.65 (1.38)      | 16.54 (3.69)      | 13.72 (2.27)      | 82.82 (1.83)      |
| LICG(ResNet-18) | 41.38 (1.48)      | 22.13 (1.72)      | 45.14 (1.41)      | 40.76 (2.09)      | 17.50 (2.94)      | 85.65 (1.79)      |
| AIN             | 44.16 (1.54)      | 57.47 (1.91)      | 56.71 (1.76)      | 46.64 (5.31)      | 65.41 (3.10)      | 83.75 (2.39)      |
| LCIG(AIN)       | **50.83 (0.88)*** | **65.69 (2.80)*** | 54.59 (1.12)      | 57.63 (3.32)      | 71.29 (4.61)      | 92.08 (2.95)      |
| ED              | 48.64 (0.93)      | 62.71 (1.33)      | **66.96 (5.63)*** | 58.16 (3.02)      | **75.76 (8.08)*** | 94.74 (1.76)      |
| LCIG(ED)        | 

In [17]:
plot_table_results("id_pscore", show_individual=False)

,C3D,dSprites,MPI3D,CLEVR,I-RAVEN,Sh3D
ResNet-18,33.12 (0.64),21.30 (0.87),43.65 (1.38),20.41 (5.38),12.51 (3.75),84.78 (1.42)
LICG(ResNet-18),41.39 (1.48),22.13 (1.72),45.14 (1.41),41.93 (3.55),17.03 (1.78),85.12 (1.60)
AIN,44.16 (1.54),60.63 (2.30),56.71 (1.76),48.62 (4.66),64.82 (2.27),84.03 (2.00)
LCIG(AIN),50.83 (0.88)*,65.69 (2.80)*,54.59 (1.12),61.16 (3.17)*,69.96 (3.56),90.38 (0.96)
ED,48.21 (0.58),62.89 (0.92),66.96 (5.63)*,57.83 (5.44),76.73 (3.32)*,96.39 (2.14)*
LCIG(ED),49.34 (0.79),48.27 (3.32),59.92 (5.74),59.51 (2.43),73.13 (5.90),96.30 (2.15)


|                 | C3D               | dSprites          | MPI3D             | CLEVR             | I-RAVEN           | Sh3D              |
|:----------------|:------------------|:------------------|:------------------|:------------------|:------------------|:------------------|
| ResNet-18       | 33.12 (0.64)      | 21.30 (0.87)      | 43.65 (1.38)      | 20.41 (5.38)      | 12.51 (3.75)      | 84.78 (1.42)      |
| LICG(ResNet-18) | 41.39 (1.48)      | 22.13 (1.72)      | 45.14 (1.41)      | 41.93 (3.55)      | 17.03 (1.78)      | 85.12 (1.60)      |
| AIN             | 44.16 (1.54)      | 60.63 (2.30)      | 56.71 (1.76)      | 48.62 (4.66)      | 64.82 (2.27)      | 84.03 (2.00)      |
| LCIG(AIN)       | **50.83 (0.88)*** | **65.69 (2.80)*** | 54.59 (1.12)      | **61.16 (3.17)*** | 69.96 (3.56)      | 90.38 (0.96)      |
| ED              | 48.21 (0.58)      | 62.89 (0.92)      | **66.96 (5.63)*** | 57.83 (5.44)      | **76.73 (3.32)*** | **96.39 (2.14)*** |
| LCIG(ED)        | 

In [7]:
plot_table_results("oracle", show_individual=True)

,C3D,dSprites,MPI3D,CLEVR,I-RAVEN,Sh3D
ResNet-18,34.54 (0.65),22.52 (1.05),44.27 (1.09),27.57 (6.55),15.54 (2.15),84.83 (1.38)
LICG(ResNet-18),54.83 (1.28),24.26 (1.52),46.68 (0.61),48.27 (5.13),33.38 (3.43),92.72 (1.54)
LICG(ResNet-18)-64,39.83 (1.57),19.62 (1.26),46.04 (0.63),44.30 (3.82),19.04 (2.22),85.30 (2.48)
LICG(ResNet-18)-128,42.94 (0.87),24.26 (1.52),46.28 (1.09),44.01 (2.72),22.25 (3.92),86.71 (2.48)
LICG(ResNet-18)-256,43.14 (1.41),23.54 (3.32),46.68 (0.61),48.27 (5.13),23.53 (5.17),86.76 (1.31)
LICG(ResNet-18)-ALG,54.83 (1.28),25.14 (2.48),46.58 (1.22),46.69 (4.54),33.38 (3.43),92.72 (1.54)
AIN,46.11 (1.10),61.09 (2.36),59.99 (3.85),61.04 (2.18),71.80 (4.47),87.20 (4.79)
LCIG(AIN),60.91 (2.09)*,68.64 (—)*,56.34 (—),69.54 (—)*,82.96 (2.38),94.70 (—)
LICG(AIN)-64,56.19 (2.62),67.48 (2.71),62.12 (6.69),65.07 (2.59),75.46 (5.50),94.70 (—)
LICG(AIN)-128,55.99 (0.94),68.64 (—)*,54.55 (—),69.54 (—)*,82.96 (2.38),95.90 (1.44)


|                     | C3D               | dSprites       | MPI3D         | CLEVR          | I-RAVEN           | Sh3D              |
|:--------------------|:------------------|:---------------|:--------------|:---------------|:------------------|:------------------|
| ResNet-18           | 34.54 (0.65)      | 22.52 (1.05)   | 44.27 (1.09)  | 27.57 (6.55)   | 15.54 (2.15)      | 84.83 (1.38)      |
| LICG(ResNet-18)     | 54.83 (1.28)      | 24.26 (1.52)   | 46.68 (0.61)  | 48.27 (5.13)   | 33.38 (3.43)      | 92.72 (1.54)      |
| LICG(ResNet-18)-64  | 39.83 (1.57)      | 19.62 (1.26)   | 46.04 (0.63)  | 44.30 (3.82)   | 19.04 (2.22)      | 85.30 (2.48)      |
| LICG(ResNet-18)-128 | 42.94 (0.87)      | 24.26 (1.52)   | 46.28 (1.09)  | 44.01 (2.72)   | 22.25 (3.92)      | 86.71 (2.48)      |
| LICG(ResNet-18)-256 | 43.14 (1.41)      | 23.54 (3.32)   | 46.68 (0.61)  | 48.27 (5.13)   | 23.53 (5.17)      | 86.76 (1.31)      |
| LICG(ResNet-18)-ALG | 54.83 (1.28)      | 25.14 (2.48)   | 4

## Ablación MPI3d sobre cantidad de capas a considerar en AIN

In [65]:
import re
import pandas as pd
import numpy as np

# =========================
# CONFIG
# =========================
DATASET = "mpi3d"
DECIMALS = 2
BOLD_TOL = 1e-12

# Debe existir en tu notebook/script
# METRIC_COLS = ["train_acc", "val_acc", "ood_val_0_acc", "test_acc"]

TARGET_ARCHS = {
    "split",          # profundidad 0
    "split_1",
    "split_2",
    "split_3",
    "split_4",
    "split_resnet_mixer_red_64",   # profundidad 0
    "split_resnet_mixer_red_128",  # profundidad 0
    "split_resnet_mixer_red_256",  # profundidad 0
    "split_resnet_mixer_alg_s1",
    "split_resnet_mixer_alg_s2",
    "split_resnet_mixer_alg_s3",
    "split_resnet_mixer_alg_s4",
    "split_resnet_algebraic_non_iid", 
    *{
        f"split_resnet_mixer_s{s}_rp{rp}_all_cases"
        for s in [1, 2, 3, 4]
        for rp in [64, 128, 256]
    },
}

ROW_ORDER = [
    "AIN",
    "LCIG(AIN)-TF",
    "LCIG(AIN)-ReducedRep (64)",
    "LCIG(AIN)-ReducedRep (128)",
    "LCIG(AIN)-ReducedRep (256)",
    "LATTICE(AIN)-ALG",
]

COL_ORDER = ["0", "1", "2", "3", "4"]  # profundidad

def fmt(mean, std, decimals=2):
    if pd.isna(mean):
        return ""
    if pd.isna(std):
        return f"{mean:.{decimals}f}"
    return f"{mean:.{decimals}f} ({std:.{decimals}f})"

def parse_arch(arch: str):
    """
    Convierte el nombre del arch en:
      - row_label
      - depth
    """
    arch = str(arch)

    # AIN profundidad 0
    if arch == "split":
        return "AIN", "0"
    if arch == "split_resnet_algebraic_non_iid":
        return "LATTICE(AIN)-ALG", "0"
    # AIN profundidad 1..4
    m = re.fullmatch(r"split_(\d+)", arch)
    if m:
        depth = m.group(1)
        return "AIN", depth

    # ReducedRep profundidad 0
    m = re.fullmatch(r"split_resnet_mixer_red_(\d+)", arch)
    if m:
        rp = int(m.group(1))
        return f"LCIG(AIN)-ReducedRep ({rp})", "0"

    # ReducedRep profundidad 1..4
    m = re.fullmatch(r"split_resnet_mixer_s(\d+)_rp(\d+)_all_cases", arch)
    if m:
        depth = m.group(1)
        rp = int(m.group(2))
        return f"LCIG(AIN)-ReducedRep ({rp})", depth
    # ALG profundidad 1..4
    m = re.fullmatch(r"split_resnet_mixer_alg_s(\d+)", arch)
    if m:
        depth = m.group(1)
        return "LATTICE(AIN)-ALG", depth
    return None, None

def plot_ablation_results(dataset, method):
    # =========================
    # CARGA Y FILTRADO
    # =========================
    df = pd.read_pickle(f"{dataset}_{method}.pkl").copy()
    
    if "arch" not in df.columns or "seed" not in df.columns or "test_acc" not in df.columns:
        raise ValueError(f"{dataset}_id.pkl debe contener columnas: arch, seed, test_acc")
    
    df = df[df["arch"].isin(TARGET_ARCHS)].copy()
    
    if df.empty:
        raise ValueError(f"No se encontraron modelos target en {dataset}_{method}.pkl")
    
    # =========================
    # MEJOR CONFIG POR ARCH
    # =========================
    non_metric_cols = [c for c in df.columns if c not in METRIC_COLS]
    config_cols = [c for c in non_metric_cols if c != "seed"]
    
    # 1) promedio por seed dentro de cada configuración
    per_seed = (
        df.groupby(config_cols + ["seed"], dropna=False)["test_acc"]
          .mean()
          .reset_index()
    )
    
    # 2) mean/std entre seeds por configuración
    stats = (
        per_seed.groupby(config_cols, dropna=False)["test_acc"]
        .agg(mean="mean", std="std", n="count")
        .reset_index()
    )
    
    # 3) mejor configuración por arch
    #  best_idx = stats.groupby("arch")["mean"].idxmax()
    # best = stats.loc[best_idx].copy()
    stats["score"] = stats["mean"] - stats["std"].fillna(0.0)

    best_idx = stats.groupby("arch")["score"].idxmax()
    best = stats.loc[best_idx].copy()
    # =========================
    # PARSEAR A FILAS/COLUMNAS
    # =========================
    parsed = best["arch"].apply(parse_arch)
    best["row"] = parsed.apply(lambda x: x[0])
    best["depth"] = parsed.apply(lambda x: x[1])
    
    best = best[best["row"].notna() & best["depth"].notna()].copy()
    # =========================
    # FILA AGREGADA: mejor ReducedRep por profundidad
    # =========================
    reduced_all = best[
        best["row"].isin([
            "LCIG(AIN)-ReducedRep (64)",
            "LCIG(AIN)-ReducedRep (128)",
            "LCIG(AIN)-ReducedRep (256)",
        ])
    ].copy()
    
    if not reduced_all.empty:
        best_reduced_by_depth_idx = reduced_all.groupby("depth")["score"].idxmax()
        best_reduced_by_depth = reduced_all.loc[best_reduced_by_depth_idx].copy()
    
        best_reduced_by_depth["row"] = "LCIG(AIN)-TF"
    
        best = pd.concat(
            [best, best_reduced_by_depth],
            ignore_index=True
        )
    # =========================
    # TABLAS NUMÉRICAS
    # =========================
    mean_df = best.pivot(index="row", columns="depth", values="mean")
    std_df  = best.pivot(index="row", columns="depth", values="std")
    
    mean_df = mean_df.reindex(index=ROW_ORDER, columns=COL_ORDER)
    std_df  = std_df.reindex(index=ROW_ORDER, columns=COL_ORDER)
    print(std_df)
    # máximos por profundidad
    max_by_col = mean_df.max(axis=0, skipna=True)
    is_max = mean_df.sub(max_by_col, axis=1).abs() <= BOLD_TOL
    
    # tabla base como strings
    plain = pd.DataFrame(index=mean_df.index, columns=mean_df.columns, dtype=object)
    for row in plain.index:
        for col in plain.columns:
            plain.loc[row, col] = fmt(mean_df.loc[row, col], std_df.loc[row, col], DECIMALS)
    
    # =========================
    # DISPLAY NOTEBOOK
    # =========================
    display(
        plain.style.apply(
            lambda col: [
                "font-weight: bold" if bool(is_max.loc[idx, col.name]) else ""
                for idx in col.index
            ],
            axis=0,
        )
    )
    
    # =========================
    # MARKDOWN
    # =========================
    md = plain.copy()
    for row in md.index:
        for col in md.columns:
            if bool(is_max.loc[row, col]) and md.loc[row, col] != "":
                md.loc[row, col] = f"**{md.loc[row, col]}**"
    
    print(md.to_markdown())
    
    # =========================
    # LATEX
    # =========================
    tex = plain.copy()
    for row in tex.index:
        for col in tex.columns:
            if bool(is_max.loc[row, col]) and tex.loc[row, col] != "":
                tex.loc[row, col] = r"\textbf{" + tex.loc[row, col] + "}"
    
    tex.index.name = None
    tex.columns.name = None
    
    latex_tabular = tex.to_latex(escape=False, index=True)
    
    latex_small = (
        r"\begin{table}[t]" "\n"
        r"\centering" "\n"
        r"\small" "\n"
        r"\setlength{\tabcolsep}{4pt}" "\n"
        r"\renewcommand{\arraystretch}{1.1}" "\n"
        r"\resizebox{0.8\textwidth}{!}{%" "\n"
        + latex_tabular + "\n"
        r"}" "\n"
        r"\caption{MPI3D results for AIN and LCIG(AIN)-ReducedRep variants across depths, including depth 0 baselines.}" "\n"
        r"\label{tab:mpi3d_depth_results}" "\n"
        r"\end{table}"
    )
    
    print(latex_small)

In [66]:
plot_ablation_results("shapes3d","id")

depth                              0   1   2   3   4
row                                                 
AIN                         5.392616 NaN NaN NaN NaN
LCIG(AIN)-TF                1.795374 NaN NaN NaN NaN
LCIG(AIN)-ReducedRep (64)   1.795374 NaN NaN NaN NaN
LCIG(AIN)-ReducedRep (128)  3.327249 NaN NaN NaN NaN
LCIG(AIN)-ReducedRep (256)  2.499890 NaN NaN NaN NaN
LATTICE(AIN)-ALG            3.238721 NaN NaN NaN NaN


depth,0,1,2,3,4
row,,,,,
AIN,85.19 (5.39),,,,
LCIG(AIN)-TF,92.24 (1.80),,,,
LCIG(AIN)-ReducedRep (64),92.24 (1.80),,,,
LCIG(AIN)-ReducedRep (128),92.18 (3.33),,,,
LCIG(AIN)-ReducedRep (256),92.88 (2.50),,,,
LATTICE(AIN)-ALG,90.79 (3.24),91.74,85.94,88.89,88.35


| row                        | 0                | 1         | 2         | 3         | 4         |
|:---------------------------|:-----------------|:----------|:----------|:----------|:----------|
| AIN                        | 85.19 (5.39)     |           |           |           |           |
| LCIG(AIN)-TF               | 92.24 (1.80)     |           |           |           |           |
| LCIG(AIN)-ReducedRep (64)  | 92.24 (1.80)     |           |           |           |           |
| LCIG(AIN)-ReducedRep (128) | 92.18 (3.33)     |           |           |           |           |
| LCIG(AIN)-ReducedRep (256) | **92.88 (2.50)** |           |           |           |           |
| LATTICE(AIN)-ALG           | 90.79 (3.24)     | **91.74** | **85.94** | **88.89** | **88.35** |
\begin{table}[t]
\centering
\small
\setlength{\tabcolsep}{4pt}
\renewcommand{\arraystretch}{1.1}
\resizebox{0.8\textwidth}{!}{%
\begin{tabular}{llllll}
\toprule
 & 0 & 1 & 2 & 3 & 4 \\
\midrule
AIN & 85.19 (5.39) &  

In [46]:
DATASET="iraven"
method="id"
df = pd.read_pickle(f"{DATASET}_{method}.pkl")
print([a for a in sorted(df["arch"].unique()) if "alg" in a])

['ed_algebraic_non_iid', 'lattice_ain_alg_l_0.5', 'lattice_ain_alg_l_0.75', 'lattice_ain_alg_l_1.5', 'lattice_ain_alg_l_2', 'resnet18_algebraic_non_iid', 'split_resnet_algebraic_adv', 'split_resnet_algebraic_iid', 'split_resnet_algebraic_non_iid', 'split_resnet_algebraic_non_iid_unpredictable_target_1', 'split_resnet_algebraic_non_iid_unpredictable_target_2', 'split_resnet_mixer_alg_s1', 'split_resnet_mixer_alg_s2', 'split_resnet_mixer_alg_s3', 'split_resnet_mixer_alg_s4']


In [47]:
df = pd.read_pickle(f"{DATASET}_{method}.pkl")

alg = [a for a in sorted(df["arch"].unique()) if "alg_s" in str(a)]
print("1. en el pkl:      ", alg)
print("2. en TARGET_ARCHS:", [a for a in alg if a in TARGET_ARCHS])
print("3. parse_arch:     ", {a: parse_arch(a) for a in alg})

1. en el pkl:       ['split_resnet_mixer_alg_s1', 'split_resnet_mixer_alg_s2', 'split_resnet_mixer_alg_s3', 'split_resnet_mixer_alg_s4']
2. en TARGET_ARCHS: ['split_resnet_mixer_alg_s1', 'split_resnet_mixer_alg_s2', 'split_resnet_mixer_alg_s3', 'split_resnet_mixer_alg_s4']
3. parse_arch:      {'split_resnet_mixer_alg_s1': ('LATTICE(AIN)-ALG', '1'), 'split_resnet_mixer_alg_s2': ('LATTICE(AIN)-ALG', '2'), 'split_resnet_mixer_alg_s3': ('LATTICE(AIN)-ALG', '3'), 'split_resnet_mixer_alg_s4': ('LATTICE(AIN)-ALG', '4')}


In [18]:
plot_ablation_results("mpi3d","oracle")

depth,0,1,2,3,4
row,,,,,
AIN,55.80 (0.34),61.40 (1.07),70.84 (0.32),73.72 (0.71),65.01 (5.77)
LCIG(AIN)-ReducedRep (64),57.30 (2.30),70.01 (0.60),71.99 (0.84),72.41 (0.88),74.44 (1.82)
LCIG(AIN)-ReducedRep (128),58.75 (4.22),65.46 (5.37),73.51 (1.90),73.55 (0.95),69.19 (8.95)
LCIG(AIN)-ReducedRep (256),52.91 (4.32),59.03 (6.85),72.20 (1.90),69.57 (3.58),51.76 (17.42)


| row                        | 0                | 1                | 2                | 3                | 4                |
|:---------------------------|:-----------------|:-----------------|:-----------------|:-----------------|:-----------------|
| AIN                        | 55.80 (0.34)     | 61.40 (1.07)     | 70.84 (0.32)     | **73.72 (0.71)** | 65.01 (5.77)     |
| LCIG(AIN)-ReducedRep (64)  | 57.30 (2.30)     | **70.01 (0.60)** | 71.99 (0.84)     | 72.41 (0.88)     | **74.44 (1.82)** |
| LCIG(AIN)-ReducedRep (128) | **58.75 (4.22)** | 65.46 (5.37)     | **73.51 (1.90)** | 73.55 (0.95)     | 69.19 (8.95)     |
| LCIG(AIN)-ReducedRep (256) | 52.91 (4.32)     | 59.03 (6.85)     | 72.20 (1.90)     | 69.57 (3.58)     | 51.76 (17.42)    |
\begin{table}[t]
\centering
\small
\setlength{\tabcolsep}{4pt}
\renewcommand{\arraystretch}{1.1}
\resizebox{0.8\textwidth}{!}{%
\begin{tabular}{llllll}
\toprule
 & 0 & 1 & 2 & 3 & 4 \\
\midrule
AIN & 55.80 (0.34) & 61.40 (1.07) & 70.84 (0.32) &

## Ablación LCIG

In [5]:
import pandas as pd
import numpy as np

# ============================================================
# CONFIG
# ============================================================
DECIMALS = 2
BOLD_TOL = 1e-12

# Si quieres incluir también el modelo base LCIG(AIN), descomenta la primera línea
ABLATION_ARCHES = [
    #"split_resnet_mixer_all_cases",  # LCIG(AIN) base
    "split_resnet_mixer_red_64",
    "split_resnet_mixer_red_128",
    "split_resnet_mixer_red_256",
    "split_resnet_mixer_red_iid_64",
    "split_resnet_mixer_red_iid_128",
    "split_resnet_mixer_red_iid_256",
    "split_resnet_algebraic_non_iid",
    "split_resnet_algebraic_adv",
    "split_resnet_algebraic_non_iid_unpredictable_target_1",
    "split_resnet_algebraic_non_iid_unpredictable_target_2",
    "split_resnet_algebraic_iid",
    "split_resnet_mixer_all_cases_iid",
    "split_resnet_mixer_no_mixer",
    "split_resnet_mixer_no_mixer_iid",
]

ABLATION_NAME_MAP = {
    arch: MODEL_NAME_MAP[arch]
    for arch in ABLATION_ARCHES
    if arch in MODEL_NAME_MAP
}

# Orden visible de filas
ROW_ORDER = [arch for arch in ABLATION_ARCHES if arch in ABLATION_NAME_MAP]

# ============================================================
# HELPERS
# ============================================================
def safe_score(mean, std):
    if pd.isna(mean):
        return np.nan
    return mean - (0.0 if pd.isna(std) else std)

# ============================================================
# 1) Elegir mejor config por arch y dataset usando mean - std
# ============================================================
records = []

for dataset in DATASETS:
    df = pd.read_pickle(f"{dataset}_id.pkl").copy()

    if "arch" not in df.columns or "seed" not in df.columns or "test_acc" not in df.columns:
        raise ValueError(f"{dataset}_id.pkl debe contener columnas: arch, seed, test_acc")

    # Nos quedamos solo con las arquitecturas de ablación
    df = df[df["arch"].isin(ABLATION_ARCHES)].copy()
    if df.empty:
        continue

    # Todas las columnas no métricas definen la configuración, excepto seed
    non_metric_cols = [c for c in df.columns if c not in METRIC_COLS]
    config_cols = [c for c in non_metric_cols if c != "seed"]

    # Promedio por seed dentro de cada configuración
    per_seed = (
        df.groupby(config_cols + ["seed"], dropna=False)["test_acc"]
        .mean()
        .reset_index()
    )

    # Estadísticos entre seeds para cada configuración
    stats = (
        per_seed.groupby(config_cols, dropna=False)["test_acc"]
        .agg(mean="mean", std="std", n="count")
        .reset_index()
    )

    stats["score"] = stats.apply(lambda r: safe_score(r["mean"], r["std"]), axis=1)

    # Mejor configuración por arch según mean - std
    best_idx = stats.groupby("arch")["score"].idxmax()
    best = stats.loc[best_idx].copy()

    for _, r in best.iterrows():
        records.append({
            "arch": r["arch"],
            "dataset": dataset,
            "mean": r["mean"],
            "std": r["std"],
            "score": r["score"],
        })

long_df = pd.DataFrame(records)

# ============================================================
# 2) Matrices
# ============================================================
mean_df = long_df.pivot(index="arch", columns="dataset", values="mean").reindex(
    index=ROW_ORDER, columns=DATASETS
)
std_df = long_df.pivot(index="arch", columns="dataset", values="std").reindex(
    index=ROW_ORDER, columns=DATASETS
)
score_df = long_df.pivot(index="arch", columns="dataset", values="score").reindex(
    index=ROW_ORDER, columns=DATASETS
)

# Mejor ablación por dataset (usando mean - std)
best_score_by_dataset = score_df.max(axis=0, skipna=True)
is_best = score_df.sub(best_score_by_dataset, axis=1).abs() <= BOLD_TOL

# ============================================================
# 3) Tabla base
# ============================================================
plain = pd.DataFrame(index=ROW_ORDER, columns=DATASETS, dtype=object)

for ds in DATASETS:
    for arch in ROW_ORDER:
        plain.loc[arch, ds] = fmt(
            mean_df.loc[arch, ds],
            std_df.loc[arch, ds],
            decimals=DECIMALS,
        )

# ============================================================
# 4) NOTEBOOK DISPLAY
# ============================================================
plain_disp = plain.rename(index=ABLATION_NAME_MAP, columns=DATASET_NAME_MAP)

is_best_disp = (
    is_best.rename(index=ABLATION_NAME_MAP, columns=DATASET_NAME_MAP)
    .reindex(index=plain_disp.index, columns=plain_disp.columns)
    .fillna(False)
)

display(
    plain_disp.style.apply(
        lambda col: [
            "font-weight: bold" if bool(is_best_disp.loc[idx, col.name]) else ""
            for idx in col.index
        ],
        axis=0,
    )
)

# ============================================================
# 5) MARKDOWN
# ============================================================
md = plain.copy()

for ds in DATASETS:
    for arch in ROW_ORDER:
        if bool(is_best.loc[arch, ds]) and md.loc[arch, ds] != "":
            md.loc[arch, ds] = f"**{md.loc[arch, ds]}**"

md = md.rename(index=ABLATION_NAME_MAP, columns=DATASET_NAME_MAP)
print(md.to_markdown())

# ============================================================
# 6) LATEX
# ============================================================
tex = plain.copy()

for ds in DATASETS:
    for arch in ROW_ORDER:
        if bool(is_best.loc[arch, ds]) and tex.loc[arch, ds] != "":
            tex.loc[arch, ds] = r"\textbf{" + tex.loc[arch, ds] + "}"

tex = tex.rename(index=ABLATION_NAME_MAP, columns=DATASET_NAME_MAP)

dataset_headers = [DATASET_NAME_MAP.get(ds, ds) for ds in DATASETS]
colspec = "l" + "c" * len(DATASETS)

latex_lines = []
latex_lines.append(r"\begin{table}[t]")
latex_lines.append(r"\centering")
latex_lines.append(r"\small")
latex_lines.append(r"\setlength{\tabcolsep}{4pt}")
latex_lines.append(r"\renewcommand{\arraystretch}{1.1}")
latex_lines.append(r"\resizebox{\textwidth}{!}{%")
latex_lines.append(rf"\begin{{tabular}}{{{colspec}}}")
latex_lines.append(r"\toprule")
latex_lines.append("Method & " + " & ".join(dataset_headers) + r" \\")
latex_lines.append(r"\midrule")

for arch in ROW_ORDER:
    row_name = ABLATION_NAME_MAP.get(arch, arch)
    cells = [tex.loc[row_name, DATASET_NAME_MAP.get(ds, ds)] for ds in DATASETS]
    latex_lines.append(row_name + " & " + " & ".join(cells) + r" \\")

latex_lines.append(r"\bottomrule")
latex_lines.append(r"\end{tabular}")
latex_lines.append(r"}")
latex_lines.append(r"\caption{Ablation results for LCIG(AIN). For each method, we select the best internal configuration using $\mathrm{mean} - \mathrm{std}$ on the selected model, and report test accuracy. Bold indicates the best ablation per dataset.}")
latex_lines.append(r"\label{tab:lcig_ain_ablation}")
latex_lines.append(r"\end{table}")

latex_small = "\n".join(latex_lines)
print(latex_small)

,C3D,dSprites,MPI3D,CLEVR,I-RAVEN,Sh3D
AIN + LICG -64,50.83 (0.88),65.69 (2.80),56.61 (5.00),55.06 (4.33),65.38 (2.42),92.24 (1.80)
AIN + LICG -128,51.02 (1.55),65.63 (4.09),54.59 (1.12),62.97 (3.04),69.55 (6.73),92.18 (3.33)
AIN + LICG -256,51.66 (3.03),63.68 (2.22),59.03 (7.62),58.88 (6.18),71.41 (7.91),92.88 (2.50)
AIN + LICG -64 + IID,51.36 (3.29),58.01 (2.52),55.27 (1.55),59.55 (3.11),69.32 (2.80),90.07 (2.86)
AIN + LICG -128 + IID,49.83 (2.34),60.33 (2.62),55.30 (1.59),57.05 (8.42),70.62 (4.50),90.92 (2.05)
AIN + LICG -256 + IID,50.80 (2.14),57.25 (3.98),56.30 (2.66),53.12 (13.22),71.52 (4.31),91.75 (3.11)
AIN + LICG -ALG,54.33 (2.73),67.36 (2.71),57.46 (5.99),59.76 (4.10),69.71 (8.07),90.79 (3.24)
AIN + LICG - ALG + ADV(ALL),0.24 (0.06),7.35 (0.65),0.00 (0.00),0.43 (0.29),2.02 (1.62),14.69 (1.44)
AIN + LICG - ALG + ADV (1),49.35 (1.02),62.78 (2.13),55.89 (2.05),83.97 (5.35),76.46 (5.55),86.36 (1.93)
AIN + LICG - ALG + ADV (2),49.06 (0.74),63.95 (3.03),56.54 (4.40),23.14 (0.00),67.89 (5.29),84.70 (0.07)


|                             | C3D              | dSprites         | MPI3D            | CLEVR            | I-RAVEN          | Sh3D             |
|:----------------------------|:-----------------|:-----------------|:-----------------|:-----------------|:-----------------|:-----------------|
| AIN + LICG -64              | 50.83 (0.88)     | 65.69 (2.80)     | 56.61 (5.00)     | 55.06 (4.33)     | 65.38 (2.42)     | 92.24 (1.80)     |
| AIN + LICG -128             | 51.02 (1.55)     | 65.63 (4.09)     | 54.59 (1.12)     | 62.97 (3.04)     | 69.55 (6.73)     | 92.18 (3.33)     |
| AIN + LICG -256             | 51.66 (3.03)     | 63.68 (2.22)     | 59.03 (7.62)     | 58.88 (6.18)     | 71.41 (7.91)     | 92.88 (2.50)     |
| AIN + LICG -64 + IID        | 51.36 (3.29)     | 58.01 (2.52)     | 55.27 (1.55)     | 59.55 (3.11)     | 69.32 (2.80)     | 90.07 (2.86)     |
| AIN + LICG -128 + IID       | 49.83 (2.34)     | 60.33 (2.62)     | 55.30 (1.59)     | 57.05 (8.42)     | 70.62 (4.50)    

In [9]:
import pandas as pd
import numpy as np

# ============================================================
# CONFIG
# ============================================================
DECIMALS = 2
BOLD_TOL = 1e-12

ABLATION_ARCHES = [
    #"split_resnet_mixer_all_cases",  # LCIG(AIN) base
    "split_resnet_mixer_red_64",
    "split_resnet_mixer_red_128",
    "split_resnet_mixer_red_256",
    "split_resnet_mixer_red_iid_64",
    "split_resnet_mixer_red_iid_128",
    "split_resnet_mixer_red_iid_256",
    "split_resnet_mixer_all_cases_iid",
    "split_resnet_mixer_no_mixer",
    "split_resnet_mixer_no_mixer_iid",
]

# Grupos sintéticos que quieres agregar
GROUPED_ARCHES = {
    "__licg_ain_group__": [
        "split_resnet_mixer_red_64",
        "split_resnet_mixer_red_128",
        "split_resnet_mixer_red_256",
    ],
    "__licg_ain_iid_group__": [
        "split_resnet_mixer_red_iid_64",
        "split_resnet_mixer_red_iid_128",
        "split_resnet_mixer_red_iid_256",
    ],
}

ABLATION_NAME_MAP = {
    arch: MODEL_NAME_MAP[arch]
    for arch in ABLATION_ARCHES
    if arch in MODEL_NAME_MAP
}

# Nombres visibles para los grupos sintéticos
ABLATION_NAME_MAP.update({
    "__licg_ain_group__": "LATTICE(AIN)",
    "__licg_ain_iid_group__": "LATTICE(AIN)+IID",
})

# Orden visible de filas: primero todas las originales, luego los resúmenes
ROW_ORDER = [arch for arch in ABLATION_ARCHES if arch in ABLATION_NAME_MAP] + [
    "__licg_ain_group__",
    "__licg_ain_iid_group__",
]

# ============================================================
# HELPERS
# ============================================================
def safe_score(mean, std):
    if pd.isna(mean):
        return np.nan
    return mean - (0.0 if pd.isna(std) else std)

# ============================================================
# 1) Elegir mejor config por arch y dataset usando mean - std
# ============================================================
records = []

for dataset in DATASETS:
    df = pd.read_pickle(f"{dataset}_id.pkl").copy()

    if "arch" not in df.columns or "seed" not in df.columns or "test_acc" not in df.columns:
        raise ValueError(f"{dataset}_id.pkl debe contener columnas: arch, seed, test_acc")

    # Nos quedamos solo con las arquitecturas de ablación originales
    df = df[df["arch"].isin(ABLATION_ARCHES)].copy()
    if df.empty:
        continue

    # Todas las columnas no métricas definen la configuración, excepto seed
    non_metric_cols = [c for c in df.columns if c not in METRIC_COLS]
    config_cols = [c for c in non_metric_cols if c != "seed"]

    # Promedio por seed dentro de cada configuración
    per_seed = (
        df.groupby(config_cols + ["seed"], dropna=False)["test_acc"]
        .mean()
        .reset_index()
    )

    # Estadísticos entre seeds para cada configuración
    stats = (
        per_seed.groupby(config_cols, dropna=False)["test_acc"]
        .agg(mean="mean", std="std", n="count")
        .reset_index()
    )

    stats["score"] = stats.apply(lambda r: safe_score(r["mean"], r["std"]), axis=1)

    # Mejor configuración por arch según mean - std
    best_idx = stats.groupby("arch")["score"].idxmax()
    best = stats.loc[best_idx].copy()

    for _, r in best.iterrows():
        records.append({
            "arch": r["arch"],
            "dataset": dataset,
            "mean": r["mean"],
            "std": r["std"],
            "score": r["score"],
        })

long_df = pd.DataFrame(records)

# ============================================================
# 1.5) Agregar filas agrupadas
# ============================================================
group_records = []

for grouped_arch, member_arches in GROUPED_ARCHES.items():
    subset = long_df[long_df["arch"].isin(member_arches)].copy()
    if subset.empty:
        continue

    # Para cada dataset, elegir el miembro con mayor score = mean - std
    best_idx = subset.groupby("dataset")["score"].idxmax()
    best_subset = subset.loc[best_idx].copy()

    for _, r in best_subset.iterrows():
        group_records.append({
            "arch": grouped_arch,
            "dataset": r["dataset"],
            "mean": r["mean"],
            "std": r["std"],
            "score": r["score"],
        })

if group_records:
    long_df = pd.concat([long_df, pd.DataFrame(group_records)], ignore_index=True)

# ============================================================
# 2) Matrices
# ============================================================
mean_df = long_df.pivot(index="arch", columns="dataset", values="mean").reindex(
    index=ROW_ORDER, columns=DATASETS
)
std_df = long_df.pivot(index="arch", columns="dataset", values="std").reindex(
    index=ROW_ORDER, columns=DATASETS
)
score_df = long_df.pivot(index="arch", columns="dataset", values="score").reindex(
    index=ROW_ORDER, columns=DATASETS
)

# Mejor ablación por dataset (usando mean - std)
best_score_by_dataset = score_df.max(axis=0, skipna=True)
is_best = score_df.sub(best_score_by_dataset, axis=1).abs() <= BOLD_TOL

# ============================================================
# 3) Tabla base
# ============================================================
plain = pd.DataFrame(index=ROW_ORDER, columns=DATASETS, dtype=object)

for ds in DATASETS:
    for arch in ROW_ORDER:
        plain.loc[arch, ds] = fmt(
            mean_df.loc[arch, ds],
            std_df.loc[arch, ds],
            decimals=DECIMALS,
        )

# ============================================================
# 4) NOTEBOOK DISPLAY
# ============================================================
plain_disp = plain.rename(index=ABLATION_NAME_MAP, columns=DATASET_NAME_MAP)

is_best_disp = (
    is_best.rename(index=ABLATION_NAME_MAP, columns=DATASET_NAME_MAP)
    .reindex(index=plain_disp.index, columns=plain_disp.columns)
    .fillna(False)
)

display(
    plain_disp.style.apply(
        lambda col: [
            "font-weight: bold" if bool(is_best_disp.loc[idx, col.name]) else ""
            for idx in col.index
        ],
        axis=0,
    )
)

# ============================================================
# 5) MARKDOWN
# ============================================================
md = plain.copy()

for ds in DATASETS:
    for arch in ROW_ORDER:
        if bool(is_best.loc[arch, ds]) and md.loc[arch, ds] != "":
            md.loc[arch, ds] = f"**{md.loc[arch, ds]}**"

md = md.rename(index=ABLATION_NAME_MAP, columns=DATASET_NAME_MAP)
print(md.to_markdown())

# ============================================================
# 6) LATEX
# ============================================================
tex = plain.copy()

for ds in DATASETS:
    for arch in ROW_ORDER:
        if bool(is_best.loc[arch, ds]) and tex.loc[arch, ds] != "":
            tex.loc[arch, ds] = r"\textbf{" + tex.loc[arch, ds] + "}"

tex = tex.rename(index=ABLATION_NAME_MAP, columns=DATASET_NAME_MAP)

dataset_headers = [DATASET_NAME_MAP.get(ds, ds) for ds in DATASETS]
colspec = "l" + "c" * len(DATASETS)

latex_lines = []
latex_lines.append(r"\begin{table}[t]")
latex_lines.append(r"\centering")
latex_lines.append(r"\small")
latex_lines.append(r"\setlength{\tabcolsep}{4pt}")
latex_lines.append(r"\renewcommand{\arraystretch}{1.1}")
latex_lines.append(r"\resizebox{\textwidth}{!}{%")
latex_lines.append(rf"\begin{{tabular}}{{{colspec}}}")
latex_lines.append(r"\toprule")
latex_lines.append("Method & " + " & ".join(dataset_headers) + r" \\")
latex_lines.append(r"\midrule")

for arch in ROW_ORDER:
    row_name = ABLATION_NAME_MAP.get(arch, arch)
    cells = [tex.loc[row_name, DATASET_NAME_MAP.get(ds, ds)] for ds in DATASETS]
    latex_lines.append(row_name + " & " + " & ".join(cells) + r" \\")

latex_lines.append(r"\bottomrule")
latex_lines.append(r"\end{tabular}")
latex_lines.append(r"}")
latex_lines.append(
    r"\caption{Ablation results for LCIG(AIN). "
    r"For each method, we select the best internal configuration using "
    r"$\mathrm{mean} - \mathrm{std}$ and report test accuracy. "
    r"We additionally report grouped summary rows LATTICE(AIN) and LATTICE(AIN)+IID, "
    r"which select the best member of each family according to the same criterion. "
    r"Bold indicates the best reported row per dataset.}"
)
latex_lines.append(r"\label{tab:lcig_ain_ablation}")
latex_lines.append(r"\end{table}")

latex_small = "\n".join(latex_lines)
print(latex_small)

,C3D,dSprites,MPI3D,CLEVR,I-RAVEN,Sh3D
LCIG(AIN)-64,50.83 (0.88),65.69 (2.80),56.61 (5.00),55.06 (4.33),65.38 (2.42),92.24 (1.80)
LCIG(AIN)-128,51.02 (1.55),65.63 (4.09),54.59 (1.12),62.97 (3.04),69.55 (6.73),92.18 (3.33)
LCIG(AIN)-256,51.66 (3.03),63.68 (2.22),59.03 (7.62),58.88 (6.18),71.41 (7.91),92.88 (2.50)
LCIG(AIN)-64 + IID,51.36 (3.29),58.01 (2.52),,,69.32 (2.80),90.07 (2.86)
LCIG(AIN)-128 + IID,49.83 (2.34),60.33 (2.62),,,70.62 (4.50),90.92 (2.05)
LCIG(AIN)-256 + IID,50.80 (2.14),57.25 (3.98),,,71.52 (4.31),91.75 (3.11)
LCIG(AIN) [ALL] + IID,,,,,,
LCIG(AIN) - MIXER,47.82 (0.87),60.43 (6.69),,,57.99 (5.23),89.28 (2.84)
LCIG(AIN) - MIXER + IID,49.05 (1.92),57.78 (2.52),,,60.48 (5.34),91.97 (3.71)
LICG(AIN),50.83 (0.88),65.69 (2.80),54.59 (1.12),62.97 (3.04),71.41 (7.91),92.24 (1.80)


|                         | C3D              | dSprites         | MPI3D            | CLEVR            | I-RAVEN          | Sh3D             |
|:------------------------|:-----------------|:-----------------|:-----------------|:-----------------|:-----------------|:-----------------|
| LCIG(AIN)-64            | **50.83 (0.88)** | **65.69 (2.80)** | 56.61 (5.00)     | 55.06 (4.33)     | 65.38 (2.42)     | **92.24 (1.80)** |
| LCIG(AIN)-128           | 51.02 (1.55)     | 65.63 (4.09)     | **54.59 (1.12)** | **62.97 (3.04)** | 69.55 (6.73)     | 92.18 (3.33)     |
| LCIG(AIN)-256           | 51.66 (3.03)     | 63.68 (2.22)     | 59.03 (7.62)     | 58.88 (6.18)     | 71.41 (7.91)     | 92.88 (2.50)     |
| LCIG(AIN)-64 + IID      | 51.36 (3.29)     | 58.01 (2.52)     |                  |                  | 69.32 (2.80)     | 90.07 (2.86)     |
| LCIG(AIN)-128 + IID     | 49.83 (2.34)     | 60.33 (2.62)     |                  |                  | 70.62 (4.50)     | 90.92 (2.05)     |
| LCIG

## Estudio Métricas

In [13]:
from __future__ import annotations

from pathlib import Path
import numpy as np
import pandas as pd

DATASETS = ["cars3d", "dsprites", "mpi3d", "clevr", "iraven", "shapes3d"]
SELECTION_METHOD = "id"  # id, ood, wio, oracle

CORE_METRIC_COLS = [
    "train_acc", "val_acc", "ood_val_0_acc", "test_acc",
    "val_4cases_twonn_id", "val_4cases_n_components_90pct", "val_4cases_topsim",
    "val_4cases_pscore_mean", "val_4cases_sv_auc", "val_4cases_hoyer_sparsity",
    "val_4cases_embedding_dim",
]

REPRESENTATION_METRICS = [
    "val_4cases_twonn_id",
    "val_4cases_topsim",
    "val_4cases_sv_auc",
    "val_4cases_pscore_mean",
    "val_4cases_hoyer_sparsity",
    "val_4cases_n_components_90pct",
]

MODEL_NAME_MAP = {
    "resnet18": "ResNet-18",
    "resnet18_mixer": "LATTICE(ResNet-18)",
    "resnet18_mixer_rp64_all_cases": "LATTICE(ResNet-18)-64",
    "resnet18_mixer_rp128_all_cases": "LATTICE(ResNet-18)-128",
    "resnet18_mixer_rp256_all_cases": "LATTICE(ResNet-18)-256",
    "resnet18_mixer_rp512_all_cases": "LATTICE(ResNet-18)-512",
    "split": "AIN",
    "split_resnet_mixer": "LCIG(AIN)",
    "split_resnet_mixer_red_64": "LCIG(AIN)-64",
    "split_resnet_mixer_red_128": "LCIG(AIN)-128",
    "split_resnet_mixer_red_256": "LCIG(AIN)-256",
    "split_resnet_mixer_all_cases": "LCIG(AIN) [ALL]",
    "ed": "ED",
    "ed_mixer_rp64": "LATTICE(ED)-64",
    "ed_mixer_rp128": "LATTICE(ED)-128",
    "ed_mixer_rp256": "LATTICE(ED)-256",
}


def safe_score(mean: float, std: float) -> float:
    if pd.isna(mean):
        return np.nan
    return mean - (0.0 if pd.isna(std) else std)


def _ensure_metric_cols(df: pd.DataFrame) -> list[str]:
    metric_cols = [c for c in CORE_METRIC_COLS if c in df.columns]
    extra_numeric = [
        c for c in df.columns
        if c not in metric_cols
        and pd.api.types.is_numeric_dtype(df[c])
        and c not in {"seed", "n_epoch"}
    ]
    return list(dict.fromkeys(metric_cols + extra_numeric))


def _build_mask(df: pd.DataFrame, row: pd.Series, cols: list[str]) -> pd.Series:
    mask = pd.Series(True, index=df.index)
    for col in cols:
        val = row[col]
        if pd.isna(val):
            mask &= df[col].isna()
        else:
            mask &= (df[col] == val)
    return mask


def select_best_config_per_model_and_dataset(
    method: str = SELECTION_METHOD,
    datasets: list[str] = DATASETS,
    representation_metrics: list[str] = REPRESENTATION_METRICS,
    config_cols: list[str] | None = None,
) -> pd.DataFrame:
    rows = []

    for dataset in datasets:
        pkl_path = Path(f"{dataset}_{method}.pkl")
        if not pkl_path.exists():
            print(f"[WARN] No existe: {pkl_path}")
            continue

        df = pd.read_pickle(pkl_path).copy()
        if df.empty:
            continue

        required = {"arch", "seed", "test_acc"}
        missing = required - set(df.columns)
        if missing:
            raise ValueError(f"{pkl_path} sin columnas requeridas: {missing}")

        metric_cols = _ensure_metric_cols(df)

        # Mejor: pasar config_cols explícitas.
        # Si no se pasan, hacemos fallback automático.
        if config_cols is None:
            inferred_config_cols = [c for c in df.columns if c not in metric_cols and c != "seed"]
        else:
            inferred_config_cols = [c for c in config_cols if c in df.columns]

        if "arch" not in inferred_config_cols:
            inferred_config_cols = ["arch"] + inferred_config_cols

        # Promedio por seed dentro de cada config
        per_seed = (
            df.groupby(inferred_config_cols + ["seed"], dropna=False)
              .mean(numeric_only=True)
              .reset_index()
        )

        # Estadísticas de test_acc por config
        stats = (
            per_seed.groupby(inferred_config_cols, dropna=False)["test_acc"]
            .agg(test_acc_mean="mean", test_acc_std="std", n_seeds="count")
            .reset_index()
        )
        stats["score"] = stats.apply(
            lambda r: safe_score(r["test_acc_mean"], r["test_acc_std"]), axis=1
        )

        # Elegir la mejor config para CADA arch
        best_cfg_idx = stats.groupby("arch")["score"].idxmax()
        best_per_arch = stats.loc[best_cfg_idx].copy()

        # Para cada arch, recuperar sus seeds y promediar representation metrics
        for _, best_cfg in best_per_arch.iterrows():
            mask = _build_mask(per_seed, best_cfg, inferred_config_cols)
            chosen = per_seed.loc[mask].copy()

            row = {
                "dataset": dataset,
                "arch": best_cfg["arch"],
                "arch_display": MODEL_NAME_MAP.get(best_cfg["arch"], best_cfg["arch"]),
                "test_acc_mean": best_cfg["test_acc_mean"],
                "test_acc_std": best_cfg["test_acc_std"],
                "score_mean_minus_std": best_cfg["score"],
                "n_seeds": int(best_cfg["n_seeds"]),
            }

            for metric in representation_metrics:
                if metric in chosen.columns:
                    row[f"{metric}_mean"] = chosen[metric].mean()
                    row[f"{metric}_std"] = chosen[metric].std()
                else:
                    row[f"{metric}_mean"] = np.nan
                    row[f"{metric}_std"] = np.nan

            rows.append(row)

    out = pd.DataFrame(rows)

    if not out.empty:
        out = out.sort_values(["dataset", "arch_display"]).reset_index(drop=True)

    return out

In [14]:

COLUMN_NAME_MAP = {
    "dataset": "Data",
    "arch": "Arch",
    "arch_display": "Model",
    "test_acc_mean": "Acc",
    "test_acc_std": "Acc SD",
    "score_mean_minus_std": "Score",
    "n_seeds": "Seeds",

    "val_4cases_twonn_id_mean": "TwoNN",
    "val_4cases_twonn_id_std": "TwoNN SD",

    "val_4cases_topsim_mean": "TopSim",
    "val_4cases_topsim_std": "TopSim SD",

    "val_4cases_sv_auc_mean": "SV-AUC",
    "val_4cases_sv_auc_std": "SV-AUC SD",

    "val_4cases_pscore_mean_mean": "P-Score",
    "val_4cases_pscore_mean_std": "P-Score SD",

    "val_4cases_hoyer_sparsity_mean": "Hoyer",
    "val_4cases_hoyer_sparsity_std": "Hoyer SD",

    "val_4cases_n_components_90pct_mean": "NC90",
    "val_4cases_n_components_90pct_std": "NC90 SD",
}
rep_df = select_best_config_per_model_and_dataset()
rep_df
cols = [
    "dataset", "arch_display",
    "val_4cases_twonn_id_mean",
    "val_4cases_topsim_mean",
    "val_4cases_sv_auc_mean",
    "val_4cases_pscore_mean_mean",
    "val_4cases_hoyer_sparsity_mean",
    "val_4cases_n_components_90pct_mean",
]


summary_df = rep_df[cols].copy()
summary_df = summary_df.rename(columns=COLUMN_NAME_MAP)
display(summary_df)


,Data,Model,TwoNN,TopSim,SV-AUC,P-Score,Hoyer,NC90
0,cars3d,AIN,2.365693,0.083476,0.965015,0.623104,0.472316,65.4
1,cars3d,ED,4.640202,0.077667,0.981704,0.846317,0.479913,52.0
2,cars3d,LCIG(AIN)-128,2.654597,0.091629,0.973774,0.884057,0.173394,49.0
3,cars3d,LCIG(AIN)-256,2.269419,0.103651,0.969905,0.877703,0.153427,55.6
4,cars3d,LCIG(AIN)-64,2.707836,0.093831,0.978961,0.894814,0.172588,37.6
...,...,...,...,...,...,...,...,...
141,shapes3d,split_resnet_mixer_no_mixer,0.000000,0.000000,0.000000,0.000000,0.000000,0.0
142,shapes3d,split_resnet_mixer_no_mixer_iid,0.000000,0.000000,0.000000,0.000000,0.000000,0.0
143,shapes3d,split_resnet_mixer_red_iid_128,0.000000,0.000000,0.000000,0.000000,0.000000,0.0
144,shapes3d,split_resnet_mixer_red_iid_256,0.000000,0.000000,0.000000,0.000000,0.000000,0.0


In [16]:
import re
import pandas as pd
import numpy as np

# ============================================================
# CONFIG
# ============================================================
DECIMALS = 3

# Orden de datasets
DATASET_ORDER = ["cars3d", "dsprites", "mpi3d", "clevr", "iraven", "shapes3d"]

# Nombres visibles de datasets (opcional)
DATASET_NAME_MAP = {
    "cars3d": "Cars3D",
    "dsprites": "dSprites",
    "mpi3d": "MPI3D",
    "clevr": "CLEVR",
    "iraven": "I-RAVEN",
    "shapes3d": "Shapes3D",
}

# Métricas a reportar: una tabla por cada una
METRIC_SPECS = [
    ("val_4cases_twonn_id_mean", "TwoNN"),
    ("val_4cases_topsim_mean", "TopSim"),
    ("val_4cases_sv_auc_mean", "SV-AUC"),
    ("val_4cases_pscore_mean_mean", "P-Score"),
    ("val_4cases_hoyer_sparsity_mean", "Hoyer"),
    ("val_4cases_n_components_90pct_mean", "NC90"),
]

# Orden pensado para comparar familias:
# base -> LATTICE(base) -> variantes de esa familia
MODEL_ORDER = [
    "resnet18",
    "resnet18_mixer",
    "resnet18_mixer_rp64_all_cases",
    "resnet18_mixer_rp128_all_cases",
    "resnet18_mixer_rp256_all_cases",
    "resnet18_mixer_rp512_all_cases",

    "split",
    "split_resnet_mixer",
    "split_resnet_mixer_red_64",
    "split_resnet_mixer_red_128",
    "split_resnet_mixer_red_256",
    "split_resnet_mixer_all_cases",

    "ed",
    "ed_mixer_rp64",
    "ed_mixer_rp128",
    "ed_mixer_rp256",
]

# Si ya tienes MODEL_NAME_MAP definido antes, reutilízalo.
# Si no, asegúrate de que exista en tu notebook.


# ============================================================
# HELPERS
# ============================================================
def _natural_suffix_key(s: str):
    """
    Orden auxiliar por si aparece algún modelo no listado en MODEL_ORDER.
    Intenta ordenar por familia y por número final si existe.
    """
    m = re.search(r"(\d+)(?!.*\d)", str(s))
    num = int(m.group(1)) if m else -1
    return (str(s), num)


def prepare_rep_df(rep_df: pd.DataFrame) -> pd.DataFrame:
    df = rep_df.copy()

    # nombres visibles
    if "arch_display" not in df.columns:
        df["arch_display"] = df["arch"].map(MODEL_NAME_MAP).fillna(df["arch"])

    df["dataset"] = pd.Categorical(df["dataset"], categories=DATASET_ORDER, ordered=True)

    # orden explícito de modelos
    present_arches = df["arch"].dropna().unique().tolist()

    known_arches = [m for m in MODEL_ORDER if m in present_arches]
    unknown_arches = sorted([m for m in present_arches if m not in MODEL_ORDER], key=_natural_suffix_key)

    final_arch_order = known_arches + unknown_arches

    df["arch"] = pd.Categorical(df["arch"], categories=final_arch_order, ordered=True)

    # para mostrar
    display_order = [MODEL_NAME_MAP.get(a, a) for a in final_arch_order]
    df["arch_display"] = pd.Categorical(df["arch_display"], categories=display_order, ordered=True)

    return df.sort_values(["arch", "dataset"]).reset_index(drop=True)


def make_metric_table(
    rep_df: pd.DataFrame,
    metric_col: str,
    metric_name: str | None = None,
    decimals: int = DECIMALS,
    rename_datasets: bool = True,
    add_average: bool = True,
    avg_col_name: str = "Avg.",
) -> pd.DataFrame:
    df = prepare_rep_df(rep_df)

    table = (
        df.pivot(index="arch_display", columns="dataset", values=metric_col)
          .reindex(index=df["arch_display"].cat.categories)
    )

    # sacar filas completamente vacías
    table = table.dropna(how="all")

    # mantener orden de datasets solo con las columnas presentes
    ordered_cols = [d for d in DATASET_ORDER if d in table.columns]
    table = table[ordered_cols]

    # promedio por modelo sobre datasets disponibles
    if add_average:
        table[avg_col_name] = table[ordered_cols].mean(axis=1, skipna=True)

    if rename_datasets:
        table = table.rename(columns=DATASET_NAME_MAP)

    return table.round(decimals)


def make_all_metric_tables(rep_df: pd.DataFrame) -> dict[str, pd.DataFrame]:
    tables = {}
    for metric_col, metric_name in METRIC_SPECS:
        if metric_col in rep_df.columns:
            tables[metric_name] = make_metric_table(rep_df, metric_col, metric_name)
    return tables


# ============================================================
# USO
# ============================================================
rep_df = select_best_config_per_model_and_dataset()

metric_tables = make_all_metric_tables(rep_df)

for metric_name, table in metric_tables.items():
    print(f"\n{'='*20} {metric_name} {'='*20}")
    display(table)


==================== TwoNN ====================


dataset,Cars3D,dSprites,MPI3D,CLEVR,I-RAVEN,Shapes3D,Avg.
ResNet-18,3.327,0.725,2.956,2.754,0.085,0.530,1.729
LICG(ResNet-18)-64,2.280,0.657,2.182,2.473,0.147,0.254,1.332
LICG(ResNet-18)-128,2.326,0.632,2.149,2.358,0.147,0.230,1.307
LICG(ResNet-18)-256,2.444,0.666,2.145,2.503,0.149,0.244,1.359
AIN,2.366,0.860,3.286,3.134,0.082,0.666,1.732
LCIG(AIN)-64,2.708,0.670,2.158,2.677,0.174,0.326,1.452
LCIG(AIN)-128,2.655,0.744,2.122,2.654,0.174,0.295,1.441
LCIG(AIN)-256,2.269,0.800,2.352,2.416,0.171,0.339,1.391
ED,4.640,0.602,2.192,1.351,0.109,0.270,1.527
LICG(ED)-64,3.110,0.549,1.620,2.632,0.107,0.219,1.373



==================== TopSim ====================


dataset,Cars3D,dSprites,MPI3D,CLEVR,I-RAVEN,Shapes3D,Avg.
ResNet-18,0.102,0.066,0.035,0.354,0.407,0.166,0.189
LICG(ResNet-18)-64,0.075,0.038,0.010,0.293,0.195,0.206,0.136
LICG(ResNet-18)-128,0.075,0.028,0.014,0.290,0.208,0.257,0.145
LICG(ResNet-18)-256,0.088,0.029,0.025,0.304,0.210,0.252,0.151
AIN,0.083,0.176,0.071,0.334,0.462,0.290,0.236
LCIG(AIN)-64,0.094,0.066,0.032,0.298,0.149,0.219,0.143
LCIG(AIN)-128,0.092,0.070,0.039,0.306,0.157,0.234,0.150
LCIG(AIN)-256,0.104,0.097,0.029,0.310,0.150,0.206,0.149
ED,0.078,0.182,0.059,0.314,0.421,0.365,0.236
LICG(ED)-64,0.073,0.149,0.040,0.241,0.365,0.284,0.192



==================== SV-AUC ====================


...

dataset,Cars3D,dSprites,MPI3D,CLEVR,I-RAVEN,Shapes3D,Avg.
ResNet-18,0.534,0.812,0.524,0.868,0.857,0.811,0.734
LICG(ResNet-18)-64,0.892,0.989,0.783,1.000,0.994,0.999,0.943
LICG(ResNet-18)-128,0.894,0.993,0.789,0.999,0.995,1.000,0.945
LICG(ResNet-18)-256,0.899,0.962,0.802,0.999,0.995,0.999,0.943
AIN,0.623,0.929,0.737,0.951,0.975,0.899,0.852
LCIG(AIN)-64,0.895,0.990,0.733,0.999,0.997,0.990,0.934
LCIG(AIN)-128,0.884,0.979,0.721,0.999,0.998,0.994,0.929
LCIG(AIN)-256,0.878,0.968,0.679,0.998,0.990,0.995,0.918
ED,0.846,0.988,0.922,0.994,0.996,0.999,0.958
LICG(ED)-64,0.860,0.994,0.936,0.999,0.995,0.999,0.964



==================== Hoyer ====================


dataset,Cars3D,dSprites,MPI3D,CLEVR,I-RAVEN,Shapes3D,Avg.
ResNet-18,0.422,0.420,0.422,0.398,0.424,0.386,0.412
LICG(ResNet-18)-64,0.115,0.092,0.168,0.068,0.126,0.028,0.099
LICG(ResNet-18)-128,0.118,0.096,0.192,0.067,0.126,0.031,0.105
LICG(ResNet-18)-256,0.128,0.123,0.203,0.065,0.120,0.034,0.112
AIN,0.472,0.472,0.624,0.315,0.343,0.357,0.431
LCIG(AIN)-64,0.173,0.153,0.226,0.235,0.124,0.119,0.172
LCIG(AIN)-128,0.173,0.163,0.213,0.202,0.118,0.100,0.162
LCIG(AIN)-256,0.153,0.128,0.191,0.206,0.129,0.038,0.141
ED,0.480,0.388,0.441,0.446,0.396,0.409,0.427
LICG(ED)-64,0.523,0.439,0.480,0.493,0.410,0.402,0.458



==================== NC90 ====================


dataset,Cars3D,dSprites,MPI3D,CLEVR,I-RAVEN,Shapes3D,Avg.
ResNet-18,61.4,29.0,47.8,10.8,12.2,31.6,32.133
LICG(ResNet-18)-64,24.2,22.4,33.6,10.0,12.0,22.8,20.833
LICG(ResNet-18)-128,31.4,24.6,39.2,10.0,12.8,27.0,24.167
LICG(ResNet-18)-256,37.6,27.0,40.2,10.0,13.0,27.6,25.900
AIN,65.4,45.0,53.0,10.2,13.0,28.2,35.800
LCIG(AIN)-64,37.6,31.4,47.2,9.6,13.2,28.8,27.967
LCIG(AIN)-128,49.0,37.6,50.2,10.0,13.8,29.8,31.733
LCIG(AIN)-256,55.6,43.0,64.8,9.0,14.8,30.6,36.300
ED,52.0,30.0,29.2,9.2,11.6,25.8,26.300
LICG(ED)-64,17.0,21.2,20.2,6.0,9.6,23.0,16.167
